In [286]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [287]:
CSV_PATH="dataset/raw_wholesale_customers.csv"
df=pd.read_csv(CSV_PATH)
print(df.head)
RANDOM_STATE = 42
K = 3


<bound method NDFrame.head of      Channel  Region  Fresh   Milk  Grocery  Frozen  Detergents_Paper  \
0          2       3  12669   9656     7561     214              2674   
1          2       3   7057   9810     9568    1762              3293   
2          2       3   6353   8808     7684    2405              3516   
3          1       3  13265   1196     4221    6404               507   
4          2       3  22615   5410     7198    3915              1777   
..       ...     ...    ...    ...      ...     ...               ...   
435        1       3  29703  12051    16027   13135               182   
436        1       3  39228   1431      764    4510                93   
437        2       3  14531  15488    30243     437             14841   
438        1       3  10290   1981     2232    1038               168   
439        1       3   2787   1698     2510      65               477   

     Delicassen  
0          1338  
1          1776  
2          7844  
3          1788  
4  

In [288]:
#selecting our features
FEATURES=['Fresh','Milk','Grocery','Frozen','Detergents_Paper','Delicassen']
X=df[FEATURES].copy()
print(X)

     Fresh   Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0    12669   9656     7561     214              2674        1338
1     7057   9810     9568    1762              3293        1776
2     6353   8808     7684    2405              3516        7844
3    13265   1196     4221    6404               507        1788
4    22615   5410     7198    3915              1777        5185
..     ...    ...      ...     ...               ...         ...
435  29703  12051    16027   13135               182        2204
436  39228   1431      764    4510                93        2346
437  14531  15488    30243     437             14841        1867
438  10290   1981     2232    1038               168        2125
439   2787   1698     2510      65               477          52

[440 rows x 6 columns]


In [289]:

def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper
low_fresh,  high_fresh = iqr_fun(X["Fresh"])
low_milk,    high_milk = iqr_fun(X["Milk"])
low_grocery, high_grocery = iqr_fun(X["Grocery"])
low_frozen,  high_frozen = iqr_fun(X["Frozen"])
low_det,     high_det = iqr_fun(X["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(X["Delicassen"])
#clipping
X["Fresh"] = X["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)
df[FEATURES] = X

In [290]:
scaler=StandardScaler()
x_scaled=scaler.fit_transform(X)
print(x_scaled)

[[ 0.12857261  1.05158597  0.04926747 -0.95324427  0.09579175  0.06589216]
 [-0.42162716  1.08673463  0.35386453 -0.30973493  0.30651872  0.47075856]
 [-0.49064723  0.85804007  0.06793486 -0.04243744  0.38243489  2.46943983]
 ...
 [ 0.31112285  2.38267048  2.45460914 -0.86054235  2.39229863  0.55487464]
 [-0.10466425 -0.70014133 -0.7595007  -0.61070442 -0.75732904  0.7933576 ]
 [-0.84025742 -0.76473271 -0.71730938 -1.01518413 -0.65213577 -1.1228252 ]]


In [291]:
#lbow method (print SSE)

for k in range (1,11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    km.fit(x_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")


k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


In [292]:
# 5) Fit K-Means with chosen k
# --------------------------------
kmeans = KMeans(n_clusters=K, n_init="auto", random_state=RANDOM_STATE)
labels = kmeans.fit_predict(x_scaled)
df["Cluster"] = labels.astype(int)
print(df.head)

<bound method NDFrame.head of      Channel  Region     Fresh     Milk    Grocery   Frozen  Detergents_Paper  \
0          2       3  12669.00   9656.0   7561.000   214.00          2674.000   
1          2       3   7057.00   9810.0   9568.000  1762.00          3293.000   
2          2       3   6353.00   8808.0   7684.000  2405.00          3516.000   
3          1       3  13265.00   1196.0   4221.000  6404.00           507.000   
4          2       3  22615.00   5410.0   7198.000  3915.00          1777.000   
..       ...     ...       ...      ...        ...      ...               ...   
435        1       3  29703.00  12051.0  16027.000  7772.25           182.000   
436        1       3  37642.75   1431.0    764.000  4510.00            93.000   
437        2       3  14531.00  15488.0  23409.875   437.00          9419.875   
438        1       3  10290.00   1981.0   2232.000  1038.00           168.000   
439        1       3   2787.00   1698.0   2510.000    65.00           477.000  

In [293]:
# 6) Evaluate clustering
# --------------------------------
sil = silhouette_score(x_scaled, labels)
dbi = davies_bouldin_score(x_scaled, labels)
print("\n=== METRICS ===")
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")


=== METRICS ===
Silhouette Score : 0.340 (closer to +1 is better)
Davies–Bouldin   : 1.297 (lower is better)


sample_idx = [0, 1, 2]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster"]]
print("\n=== SANITY CHECK (3 Clients) ===")
print(sanity)


In [294]:
#mny second algorithm is GaussianMixture
gmm = GaussianMixture(n_components=3, covariance_type="tied", random_state=RANDOM_STATE)
gmm_labels = gmm.fit_predict(x_scaled)
df["GMM_Cluster"] = gmm_labels.astype(int)

# Evaluate using the same metrics
gmm_sil = silhouette_score(x_scaled, gmm_labels)
gmm_dbi = davies_bouldin_score(x_scaled, gmm_labels)

print("\n=== GMM METRICS ===")
print(f"Silhouette Score : {gmm_sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {gmm_dbi:.3f} (lower is better)")



=== GMM METRICS ===
Silhouette Score : 0.294 (closer to +1 is better)
Davies–Bouldin   : 1.337 (lower is better)


In [295]:
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"

print("\n=== CLUSTER CENTERS (Original Units) ===")
print(centers_df.round(2))
# GMM uses .means_ instead of .cluster_centers_
gmm_means_scaled = gmm.means_
gmm_means_original = scaler.inverse_transform(gmm_means_scaled)

gmm_centers_df = pd.DataFrame(gmm_means_original, columns=FEATURES)
gmm_centers_df.index.name = "GMM_Cluster"

print("\n=== GMM CLUSTER MEANS (Original Units) ===")
print(gmm_centers_df.round(2))


OUT_PATH = "dataset/segmented_wholesale_customers.csv"
df.to_csv(OUT_PATH, index=False)


=== CLUSTER CENTERS (Original Units) ===
            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         5869.23  10142.25  16653.32  1355.63           7022.05     1482.60
1         9418.37   2685.47   3583.98  2052.33            942.49      750.71
2        22881.83   5870.35   6773.44  5057.43           1209.38     2452.04

=== GMM CLUSTER MEANS (Original Units) ===
                Fresh      Milk   Grocery   Frozen  Detergents_Paper  \
GMM_Cluster                                                            
0             6657.36  10187.70  17296.43  1507.97           7632.50   
1            11254.56   3546.37   4683.02  1522.68           1197.06   
2            16612.40   4658.92   5188.99  6816.36            880.50   

             Delicassen  
GMM_Cluster              
0               1383.85  
1               1112.01  
2               1657.38  
